# Publisher Application

This application makes use of the Confluence Publisher class defined in this folder. See ./Publisher.py

In [ ]:
import markupsafe
header=markupsafe.Markup('<span style="color:red;"><b>Unreliable information. For testing purposes only!</b></span><br/>')

## Load configuration
Be aware not to commit your credentials!

In [ ]:
import yaml
import copy
import logging
log = logging.getLogger(__name__)

with open('private.yaml') as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

## Load the data
The **data** is the JSON serialized information model 

In [ ]:
import json

data = None
with open(config['json'], 'r') as source:
     data = json.load(source)

str(data)[:512]

In [ ]:
categories = list(data)
for category in categories:
    print('{} Elements in category "{}"'.format(len(data[category]), category))

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
default_language = config['languages'][0]
language_page_root = confluence.get_page_id(space_key, '')

In [ ]:
import Publisher as cp
publisher = cp.Publisher(config, data, confluence, space_key, root_page_id, default_language)
list(map(lambda e: (e, publisher.translate(data['entities'][e]['name'])), list(data['entities'])[:3]))

In [ ]:
result = publisher.scan_current_content()
str(result)[:200]

# Publishing

## Prepare destination structure

In [ ]:
from tqdm.notebook import tqdm_notebook
import time

total = len(publisher.content_map)
with tqdm_notebook(total=total, dynamic_ncols=True, unit='Page') as pbar:
    topic_dict = publisher.collect_recursive(config)
    topics = list(topic_dict)
    print('Processing topics ' + str(topics))
    for topic in topics:
        print('Processing ' + topic)
        entry_config = topic_dict[topic]
        
        element_title = topic
        if entry_config.get('title'):
            element_title = entry_config['title']

        folder_page_id = publisher.stub(element_title, root_page_id)
        print('Created group page "{}". Confluence page id = {}'.format(element_title, str(folder_page_id['id'])))

        for item in data[topic]:
            entry_data = data[topic][item]

            if entry_data.get('current_confluence_content'):
                pbar.update(1)
                continue

            pbar.set_description('Stubbing element {}:{}'.format(topic, item))
            title = publisher.page_title(item)

            parent = folder_page_id['id']

            # hack: nesting
            if topic == 'attributes':
                parent = publisher.page_for_key(entry_data['entity'])['pageid']

            # hack: nesting
            if topic == 'tables':
                parent = publisher.page_for_key(entry_data['interface-id+'])['pageid']

            retry = 5
            while retry > 0:
                try:
                    create_result = publisher.stub(title, parent, labels=entry_config.get('labels'))
                    page_id = create_result['id']
                    entry_data['current_confluence_content'] = create_result['current']
                    publisher.register_page_id(item, page_id)
                    retry = 0
                except Exception as e:
                    log.error('Failed to stub element {}:{}'.format(topic, item), e)
                    time.sleep(1.1)
                    retry -= 1

            pbar.update(1)
            time.sleep(.25)

## Now generate content

In [ ]:
str(publisher.content_map)[:512]

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape
import re
import time

jinja_env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)
jinja_env.globals.update({ 'util': publisher, 'header': header, })

total = len(publisher.content_map)
with tqdm_notebook(total=total, dynamic_ncols=True, unit='Page') as pbar:

    topic_dict = publisher.collect_recursive(config)
    
#    tables = topic_dict['tables']
#    topic_dict = { 'tables': tables }
    
    topics = list(topic_dict)
    for topic in topics:
        
        items = list(data[topic])    
        print('Creating content for class "{}" ({})'.format(topic, len(items)))
        entry_config = topic_dict[topic]

        jinja_template = jinja_env.get_template(entry_config['template'])

        for item in items:
            item_data = data[topic][item]
            
            item_data['icon'] = '0612'
            try:
                rendered = jinja_template.render(data=data, key=item, item=item_data)
                content_xml = re.sub('<!--.+?->(\n+)*', '', rendered) # strip comment lines
                
                minor_edit = False
                current = item_data.get('current_confluence_content')
                if current and current == content_xml:
                    minor_edit = True
                    pbar.set_description('Major update on element {}:{}'.format(topic, item))
                else:
                    pbar.set_description('Minor update on element {}:{}'.format(topic, item))
                    
                retries = 3
                while retries > 0:
                    try:
                        publisher.update_page(item, content_xml, minor_edit=minor_edit, version_comment="Automatic update")
                    except ConnectionError:
                        log.excpetion('Failed to publish, retry in 1s')
                        time.sleep(1.1)
                    finally:
                        retries -= 1

            except Exception as e:
                log.exception('Unable to process {} {}'.format(topic, item))
            time.sleep(.25)
            pbar.update(1)